In [ ]:
library(Seurat)
library(ggplot2)
library(MetaNeighbor)
library(SingleCellExperiment)
library(dplyr)

In [ ]:
library(tidyverse)
library(scales)
#library(circlize)
#library(ComplexHeatmap)
#library(pvclust)
library(stringr)

In [ ]:
hm <- readRDS('data/external/human_reference.rds')
DefaultAssay(hm) <- 'RNA'
hm@meta.data$species <- 'human'
hm

In [ ]:
merged_seurat <- readRDS("data/processed/sc_reference_file")
merged_seurat

In [ ]:
table(merged_seurat@meta.data$region)

In [ ]:
table(merged_seurat@meta.data$species)

In [ ]:
na <- subset(
  merged_seurat,
  subset = region == "PFC" & species == "NA"
)

sa <- subset(
  merged_seurat,
  subset = region == "PFC" & species == "SA"
)

# 查看提取结果（验证是否正确）
cat("NA物种 + PFC区域 细胞数：", ncol(na), "\n")
cat("SA物种 + PFC区域 细胞数：", ncol(sa), "\n")

In [ ]:
#na <- readRDS('data/processed/excitatory_neuron_file')
DefaultAssay(na) <- 'RNA'
na@meta.data$species <- 'NA'
na

In [ ]:
#sa <- readRDS('data/processed/excitatory_neuron_file')
DefaultAssay(sa) <- 'RNA'
sa@meta.data$species <- 'SA'
sa

In [ ]:
mu <- readRDS('data/external/mouse_reference.rds')
#mu <- readRDS('data/processed/cortex_analysis_file')
DefaultAssay(mu) <- 'RNA'
mu@meta.data$species <- 'mouse'
mu

In [ ]:
Idents(hm) <- hm@meta.data$cellType_layer
table(hm@meta.data$cellType_layer)

In [ ]:
Idents(na) <- na@meta.data$annotation
table(na@meta.data$annotation)

In [ ]:
Idents(sa) <- sa@meta.data$annotation
table(sa@meta.data$annotation)

In [ ]:
str(mu@meta.data)

In [ ]:
Idents(mu) <- mu@meta.data$ctx_layer
table(mu@meta.data$ctx_layer)
#Idents(mu) <- mu@meta.data$Layer
#table(mu@meta.data$Layer)

In [ ]:
sa <- subset(sa, subset = annotation != "Other")
na <- subset(na, subset = annotation != "Other")

In [ ]:
#saveRDS(sa, file = "data/processed/excitatory_neuron_file")
#saveRDS(na, file = "data/processed/excitatory_neuron_file")

In [ ]:
# 提取表达矩阵和元数据
exprs_hm <- GetAssayData(hm)
#exprs_sb <- GetAssayData(sb)
exprs_na <- GetAssayData(na)
exprs_sa <- GetAssayData(sa)
exprs_mu <- GetAssayData(mu)

meta_hm <- hm@meta.data
#meta_sb <- sb@meta.data
meta_na <- na@meta.data
meta_sa <- sa@meta.data
meta_mu <- mu@meta.data

# 提取共有基因
common_genes <- Reduce(intersect, list(rownames(exprs_hm), rownames(exprs_na), rownames(exprs_mu), rownames(exprs_sa)))

# 提取共有基因表达矩阵
exprs_hm_common <- exprs_hm[common_genes, ]
#exprs_sb_common <- exprs_sb[common_genes, ]
exprs_na_common <- exprs_na[common_genes, ]
exprs_sa_common <- exprs_sa[common_genes, ]
exprs_mu_common <- exprs_mu[common_genes, ]

# 合并表达矩阵
combined_exprs <- cbind(exprs_hm_common, exprs_na_common, exprs_mu_common, exprs_sa_common)

In [ ]:
new_colData <- data.frame(
  study_id = c(rep('HM', ncol(exprs_hm_common)), rep('NA', ncol(exprs_na_common)), rep('MU', ncol(exprs_mu_common)), rep('SA', ncol(exprs_sa_common))),
  cell_type = c(Idents(hm), Idents(na), Idents(mu), Idents(sa))
)

# 创建 SingleCellExperiment 对象
sce_combined <- SingleCellExperiment(assays = list(RNA = combined_exprs), colData = new_colData)

# 检查合并后的对象
sce_combined

In [ ]:
table(sce_combined$study_id)

In [ ]:
table(sce_combined$cell_type)

In [ ]:
var_genes <- variableGenes(dat = sce_combined, exp_labels = sce_combined$study_id)
var_genes

In [ ]:
celltype_NV <- MetaNeighborUS(var_genes = var_genes,
                              dat = sce_combined,
                              study_id = sce_combined$study_id,
                              cell_type = sce_combined$cell_type,
                              fast_version = TRUE)
head(celltype_NV)

In [ ]:
png("heatmap_integrated_4.png",width = 1200, height = 1200)
#png("heatmap_integrated_5_pf.png",width = 1200, height = 1200)
plotHeatmapPretrained(aurocs = celltype_NV, cex = 1, margins = c(16, 16))
dev.off()

In [ ]:
#pdf("heatmap_integrated_5.pdf", width = 12, height = 12)
pdf("heatmap_integrated_4.pdf", width = 12, height = 12)
#pdf("heatmap_integrated_5_pf.pdf", width = 12, height = 12)
plotHeatmapPretrained(aurocs = celltype_NV, cex = 1, margins = c(16, 16))
dev.off()

In [ ]:
colnames(celltype_NV)

In [ ]:
# 1. 加载包（确保运行）
library(MetaNeighbor)

# 2. 【关键】自定义你想要的 行/列 标签顺序
# 直接从你的 celltype_NV  colnames/rownames 里复制，然后手动排序
my_order <- c(
  
  "MU|L6b",
  "NA|EX_Nxph4_CPLX3",
  "SA|EX_Nxph4_CPLX3",  

  "MU|L6",
 "NA|EX_FOXP2",
  "SA|EX_FOXP2", 
    
      "HM|Excit_L6",
  "HM|Excit_L5/6",
    "MU|L5",
  "NA|EX_ADRA1A_Vat1l",
  "SA|EX_ADRA1A_Vat1l",
    
        "HM|Excit_L5",
      "NA|EX_TSHZ2_VWC2L",
  "SA|EX_TSHZ2_VWC2L",
    
      "NA|EX_ERG",
  "NA|EX_ERG",
     "NA|EX_Ntng2",
  "SA|EX_Ntng2", 
    
      "MU|L4",
    "NA|EX_NTNG1",
  "SA|EX_NTNG1",  
    
  "NA|EX_BDNF",
  "SA|EX_BDNF",        
  "NA|EX_ITGA4_TPBG",
  "SA|EX_ITGA4_TPBG", 
  "HM|Excit_L4",
  "HM|Excit_L3/4/5",
  "NA|EX_C1QL3",
  "SA|EX_C1QL3",
    

        
  "NA|EX_env_TMEM144",
  "SA|EX_env_TMEM144",
  "NA|EX_IFITM3_CLDN5",
  "SA|EX_IFITM3_CLDN5",
   "NA|EX_SLC1A3_FGFR3",
  "SA|EX_SLC1A3_FGFR3",
    
      "HM|Excit_L3",
  "NA|EX_HTR7_GNAL",
  "SA|EX_HTR7_GNAL",
    "MU|L2/3",   
    "HM|Excit_L2/3",  
    "MU|L2"
)

# 3. 按照自定义顺序 重排矩阵
celltype_NV_ordered <- celltype_NV[my_order, my_order]


In [ ]:
celltype_NV_ordered

In [ ]:
# 4. 画图（用重排后的矩阵）
pdf("heatmap_integrated_ordered.pdf", width = 14, height = 14)
plotHeatmapPretrained(aurocs = celltype_NV_ordered, cex = 0.8, margins = c(20, 20))
dev.off()

png("heatmap_integrated_ordered.png", width = 1400, height = 1400)
plotHeatmapPretrained(aurocs = celltype_NV_ordered, cex = 0.8, margins = c(20, 20))
dev.off()

In [ ]:
# 1. 加载pheatmap包
library(pheatmap)

# 2. 用你已经重排好的矩阵 celltype_NV_ordered
# 3. 画图：完全关闭聚类，保留你设定的顺序
pdf("heatmap_integrated_ordered_pheatmap.pdf", width = 14, height = 14)
pheatmap(
  mat = celltype_NV_ordered,
  cluster_rows = FALSE,    # 关闭行聚类
  cluster_cols = FALSE,    # 关闭列聚类
  scale = "none",          # 不做标准化，保留原始AUROC值
  display_numbers = FALSE, # 不显示数值（和原热图一致）
  fontsize = 10,
  border_color = NA,       # 去掉格子边框（和原热图一致）
  color = colorRampPalette(c("blue", "white", "red"))(100) # 匹配原热图蓝-白-红配色
)
dev.off()

In [ ]:
# 清空当前所有绘图设备（解决卡死）
graphics.off()

# 加载包
library(pheatmap)

# ===================== 必须检查 =====================
# 先确认你的矩阵还在、且正常
dim(celltype_NV_ordered)  # 必须显示 >0, >0
rownames(celltype_NV_ordered)  # 必须有标签

# ===================== 修复版绘图 =====================
pdf("heatmap_integrated_ordered_pheatmap.pdf", width = 16, height = 16)

pheatmap(
  mat = celltype_NV_ordered,
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  scale = "none",
  display_numbers = FALSE,
  fontsize = 15,
  border_color = NA,
  color = colorRampPalette(c("navy", "white", "firebrick3"))(100),
  
  # 额外加两个防止空白的参数
  treeheight_row = 0,
  treeheight_col = 0
)

dev.off()

In [ ]:
# 清空绘图设备，避免 0KB PDF
graphics.off()

library(pheatmap)

# ====================== 自动层次聚类 + 画图 ======================
pdf("heatmap_clustered.pdf", width = 16, height = 16)

pheatmap(
  # 你的原始相似矩阵（不需要手动排序！自动聚类）
  mat = celltype_NV,
  
  # ✅ 开启层次聚类（自动排序，这就是你想要的）
  cluster_rows = TRUE,
  cluster_cols = TRUE,
  
  # 不做标准化，保持 AUROC 原始值
  scale = "none",
  
  # 配色 100% 匹配你的原图：浅蓝 → 白 → 红
  color = colorRampPalette(c("navy", "white", "firebrick3"))(100),
  
  # 样式优化
  display_numbers = FALSE,
  fontsize = 12,
  border_color = NA,
  treeheight_row = 50,  # 聚类树高度
  treeheight_col = 50
)

dev.off()